In [13]:
import os 
from uuid import uuid4
from dotenv import load_dotenv
from math import sqrt
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import START , END, StateGraph
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import add_messages
from typing import TypedDict, Annotated, Literal

from pydantic import BaseModel , Field
import time
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)

In [6]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]


In [7]:
def chatnode(state: ChatState) :
    response = model.invoke(state["messages"])
    return {
        "messages" : [response]
    }

In [9]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node("chatnode", chatnode)
graph.add_edge(START, "chatnode")
graph.add_edge("chatnode", END)
chatbot = graph.compile(checkpointer = checkpointer)

In [16]:
thread_id = uuid4().hex
config = {"configurable" : {"thread_id" : thread_id}}
chatbot.invoke({"messages" : [HumanMessage(content="Hello, my name is Waleed?") ]}, config = config)

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in { "exit" , "quit", "end" }:
        break
    response = chatbot.invoke({"messages" : [HumanMessage(content=user_input)]}, config = config)
    print(f"Chatbot: {response['messages'][-1].content}")


Chatbot: I am doing well, thank you for asking! How are you?
Chatbot: Great to hear you're good, Waleed! It's nice to know your name. Is there anything I can help you with today?
Chatbot: Your name is Waleed.
Chatbot: Yes, I do! I try my best to remember information you tell me within our conversation. Is there anything else I can help you with, Waleed?
Chatbot: 10 + 15 = 25
Chatbot: 20 + 30 = 50
Chatbot: Okay, so the last result was 50.

50 * 2 = 100
100 + 1 = 101

Therefore, the answer is 101.


In [17]:
chatbot.get_state(config = config)

StateSnapshot(values={'messages': [HumanMessage(content='Hello, my name is Waleed?', additional_kwargs={}, response_metadata={}, id='29ad0b4b-4863-4a88-af9e-76b39ecb47e7'), AIMessage(content='Hello Waleed, nice to meet you! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--ae1e8a23-1400-4bf8-8c4b-ba81029406c7-0', usage_metadata={'input_tokens': 8, 'output_tokens': 17, 'total_tokens': 25, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='hello how are you', additional_kwargs={}, response_metadata={}, id='34a09e57-0f93-413f-817e-1c9793f05808'), AIMessage(content='I am doing well, thank you for asking! How are you?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='r